In [ ]:
#Importacion de librerias
import requests
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Configura los detalles de la conexión desde variables de entorno.
# Copia .env.example a .env y define los valores. Nunca escribas credenciales aquí.
import os

host     = os.getenv('DB_HOST')
port     = os.getenv('DB_PORT', '1433')
database = os.getenv('DB_NAME')

if not all([host, database]):
    raise RuntimeError(
        "Faltan variables de entorno: define DB_HOST y DB_NAME. Ver .env.example"
    )

try:
    # Crea la cadena de conexión
    url = (
        'mssql+pyodbc://@{host}:{port}/{db}'
        '?trusted_connection=yes&driver=SQL+Server'
    ).format(host=host, port=port, db=database)

    # Crear conexion con base de datos
    engine = create_engine(url)
    print("Conexion a la base de datos realizada")

except Exception as e:
    print('Error:', e)


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT 
    Identificacion,
    Periodo,
	Reintegro
    Periodo_Siguiente,
    EX_Periodo_Siguiente_Normal,
    nuevo_periodo,
    Status,
	Tipo_Salto,
    Estado_Alumno,
    Tipo_estado_alumno,
    Modalidad,
    Programa,
    Semestre_SINU,
	ciclo,
	año,
    AÑO_MEN,
	Estado_Alumno,
    anio_grado,
    Genero,
    Estado_Pago,
    RANGO_EDAD,
    RANGO_SALARIO,
    ESTA_TRABAJANDO,
    METODO_FINANCIAMIENTO,
    ZONA_RESIDENCIA,
    REGIMEN_SISTEMA_SALUD,
    PERTENECE_GRUPO_ETNICO,
    GRUPO_ETNICO,
    TIENE_DISCAPACIDAD,
    DISCAPACIDAD,
    LGBTIQ,
	ciudad,
	localidad_normalizada
FROM academico.historial_academico


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        historial_academico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT 
Identificacion, 
barrio, 
departamento, 
pais
FROM geo.ubicacion_estudiante


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        localización = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT
Periodo,
NOMBRE_GRUPO, 
IDENTIFICACION as Identificacion, 
NOMBRE_CONCEPTO, 
NOMBRE_CAUSA_NOTA
FROM becas.descuentos_beca


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        becas = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select 
IDENTIFICACION as Identificacion, 
[MATERIAS INSCRITAS], 
[MATERIAS APROBADAS], 
COD_PERIODO as Periodo, 
Porcentaje_aprobacion
FROM academico.aprobacion_materias 


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        materias = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select Periodo, Identificacion,TOTAL from 
financiera.cartera


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        finanza = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select *
from plataforma.permanencia


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        permanencia = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

# Hago copia
df = historial_academico.copy()


In [ ]:

# Verifico Columnas y Filas
df.shape


In [ ]:

# Ahora haz el merge correctamente
df1 = df.merge(
    localización,
    on='Identificacion',
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df1.shape


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
becas = becas[~becas.duplicated(subset=['Identificacion', 'Periodo'], keep='last')]


In [ ]:

# Ahora haz el merge correctamente
df2 = df1.merge(
    becas,
    on=['Identificacion','Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df2.shape


In [ ]:

# Ahora haz el merge correctamente
df3 = df2.merge(
    materias,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df3.shape


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
finanza = finanza[~finanza.duplicated(subset=['Identificacion', 'Periodo'], keep='last')]


In [ ]:

# Ahora haz el merge correctamente
df4 = df3.merge(
    finanza,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df4.shape


In [ ]:

# Paso 0: Lista de periodos a eliminar
periodos_a_excluir = [
    '24I01', '24I02', '24I03', '24I04', '24I05', '24I06',
    '24I10', '24I11', '24I12', '24I13',
    '25I01', '25I02', '25I03', '25I04', '25I11', '25I12', '25I13'
]


In [ ]:

# Paso 1: Eliminar los periodos que no se desean
permanencia = permanencia[~permanencia['cod_periodo'].isin(periodos_a_excluir)]


In [ ]:

# Paso 2: Obtener el último periodo (cod_periodo) por cada estudiante
ultimo_periodo = permanencia.groupby('num_identificacion')['cod_periodo'].max().reset_index()
ultimo_periodo.rename(columns={'cod_periodo': 'ultimo_cod_periodo'}, inplace=True)


In [ ]:

# Paso 3: Unir esta información al DataFrame original
permanenciaa = permanencia.merge(ultimo_periodo, on='num_identificacion', how='left')


In [ ]:

# Paso 4: Filtrar solo las filas del último periodo por estudiante
permanenciaa = permanenciaa[permanenciaa['cod_periodo'] == permanenciaa['ultimo_cod_periodo']]


In [ ]:

# Paso 5: Eliminar duplicados (por si un curso se repite por error)
permanenciaa = permanenciaa.drop_duplicates(subset=['num_identificacion', 'curso', 'cod_periodo'])


In [ ]:

# Paso 6: Eliminar la columna auxiliar que ya no necesitamos
permanenciaa = permanenciaa.drop(columns=['ultimo_cod_periodo'])


In [ ]:

# Paso 7: Ordenar por estudiante y curso
permanenciaa = permanenciaa.sort_values(by=['num_identificacion', 'curso'])


In [ ]:

# Paso 8: Reiniciar índices
permanenciaa = permanenciaa.reset_index(drop=True)


In [ ]:

# Resultado final:
permanenciaa


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Renombro para que me cruzen las llaves
permanenciaa = permanenciaa.rename(columns={
    'num_identificacion': 'Identificacion',
    'cod_periodo': 'Periodo'
})


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Borro las variables que no necesito
permanenciaa = permanenciaa.drop(columns=[
    'SEMESTRE', 'NOM_UNIDAD', 'NOM_DEPENDENCIA', 'CRE_PROGRAMA',
    'CREDITOS_INSCRITOS', 'Materias_BU', 'Materias_B1', 'Materias_B2',
    'nivelado_en_materias', 'itemtype', 'itemmodule', 'itemname',
    'nombrecategoria', 'peso', 'finalgrade', 'estadoactividad',
    'estudiantes', 'curso', '#ASISTENCIAS'  
    
])


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
permanenciaa = permanenciaa.drop_duplicates(subset=['Identificacion', 'Periodo'])


In [ ]:

# Verifico Columnas y Filas
df4.shape


In [ ]:

# Ahora haz el merge correctamente

df5 = df4.merge(
    permanenciaa,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df5.shape


In [ ]:
# DF final

df5


In [ ]:
# Verifico las columnas del DataFrame final

print(df5.columns.tolist())


In [ ]:
# Prototipo si quiero filtrar por Periodo y borro duplicados de identificación

#df5=df5.drop_duplicates(['Identificacion'])


#df5  = df5[df5['Periodo']=='2025B']


In [ ]:
## Verifico si hay valores nulos

df5.isnull().sum()


In [ ]:
# Verifico si hay valores nulos en la columna 'ASISTENCIA'
df5['ASISTENCIA'].value_counts(dropna=False)


In [ ]:
#Covversion de columnas sin espacios y mayusculas

df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')


In [ ]:
## Detectar columnas numéricas
#numericas = df5.select_dtypes(include=['int64', 'float64']).columns.tolist()
#
## Convertir a binario: todo valor > 0 será 1, lo demás 0
#df5_binarias = df5.copy()
#df5_binarias[numericas] = (df5_binarias[numericas] > 0).astype(int)
#
#df5_binarias.shape

In [ ]:
#Prototipo 

## Para cada columna, si el valor está presente y no es "SIN_DATO" → 1, si no → 0
#df5_binarias = df5_binarias.applymap(lambda x: 0 if pd.isna(x) or str(x).strip().upper() == "SIN_DATO" else 1)
#
#
#
## Si quieres asegurarte de que todo es int
#df5_binarias = df5_binarias.astype(int)

In [ ]:
# Prototipo si quiero crear una columna de target de deserción

#df5['TARGET_DESERCION'] = df5['STATUS'].apply(
#    lambda x: 1 if str(x).strip().lower() == 'desercion' else 0
#)

In [ ]:
# Mi columna objetivo es 'TARGET_DESERCION' y ha sido creada para identificar si un estudiante ha desertado o no y convertido ha binario.


df5['TARGET_DESERCION'] = df5['STATUS'].apply(
    lambda x: 1 if str(x).strip().lower() == 'desercion' else 0
)

In [ ]:
# Columna objetivo
col_objetivo = 'TARGET_DESERCION'


In [ ]:
# Verifico si hay valores nulos

df5.isnull().sum()

In [ ]:

# Detectar columnas numéricas (excluyendo objetivo)
numericas = df5.select_dtypes(include=['int64', 'float64']).columns.tolist()
if col_objetivo in numericas:
    numericas.remove(col_objetivo)


In [ ]:

# Detectar columnas no numéricas (excluyendo objetivo)
no_numericas = df5.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
if col_objetivo in no_numericas:
    no_numericas.remove(col_objetivo)


In [ ]:

# Copia
df5_binarias = df5.copy()


In [ ]:

# 1️⃣ Numéricas: todo >0 es 1
df5_binarias[numericas] = (df5_binarias[numericas] > 0).astype(int)


In [ ]:

# 2️⃣ No numéricas: convertir a binario por columna
def convertir_a_binario(valor):
    if pd.isna(valor):
        return 0
    valor_str = str(valor).strip().upper()
    return 0 if valor_str == "" or valor_str == "SIN_DATO" else 1

for col in no_numericas:
    df5_binarias[col] = df5_binarias[col].map(convertir_a_binario).astype(int)


In [ ]:
# Verifico las primeras filas del DataFrame resultante
print(df5_binarias.head())

In [ ]:
# Verifico las filas Y columnas del DataFrame 

df5.shape


In [ ]:
# Verifico las filas Y columnas del DataFrame resultante

df5_binarias.shape

In [ ]:
# Verifico si hay valores nulos

df5.isnull().sum()

In [ ]:
# Verifico si hay valores nulos en el DataFrame binario

df5_binarias

In [ ]:
#verifico los nombres de las columnas del DataFrame binario

df5_binarias.columns

In [ ]:
#Borro las columnas que no necesito en el DataFrame binario

df5_binarias = df5_binarias.drop(
    columns=[
        'IDENTIFICACION',
        'PERIODO_SIGUIENTE',
        'EX_PERIODO_SIGUIENTE_NORMAL',
        'NUEVO_PERIODO',
        'TIPO_ESTADO_ALUMNO',
        'PERTENECE_GRUPO_ETNICO',
        'GRUPO_ETNICO',
        'TIENE_DISCAPACIDAD',
        'DISCAPACIDAD',
        'LGBTIQ',
        'LOCALIDAD_NORMALIZADA',
        'BARRIO',
        'DEPARTAMENTO',
        'PAIS',
        'NOMBRE_GRUPO',
        'NOMBRE_CONCEPTO'
    ],
    errors='ignore'
) 

In [ ]:
df5_binarias.shape

In [ ]:
df5_binarias.columns

In [ ]:
df5_binarias

In [ ]:

# Separar X e y
X = df5_binarias.drop('TARGET_DESERCION', axis=1)
y = df5_binarias['TARGET_DESERCION']


In [ ]:

# División de datos y validación de clases
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("\nDistribución de clases en entrenamiento (original):", y_train.value_counts())


In [ ]:

# Aplicar SMOTE solo al entrenamiento
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("Distribución de clases en entrenamiento (con SMOTE):", y_train_resampled.value_counts())


In [ ]:

# Entrenamiento de modelos
modelos = {
    "Regresión Logística": LogisticRegression(random_state=42, max_iter=1000),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}


In [ ]:

for nombre, modelo in modelos.items():
    print(f"\n--- Entrenando {nombre} ---")
    modelo.fit(X_train_resampled, y_train_resampled)
    print(f"--- Evaluando {nombre} ---")
    y_pred = modelo.predict(X_test)
    print("Reporte de Clasificación:")
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matriz de Confusión - {nombre}')
    plt.ylabel('Real')
    plt.xlabel('Predicho')
    plt.show()
